In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets,Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
from collections import Counter

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}

In [3]:
# Load all English datasets
spanish_datasets = [
    load_dataset("UniversalCEFR/caes_es")["train"],
    load_dataset("UniversalCEFR/kwiqiz_es")["train"],
]
spanish_data = concatenate_datasets(spanish_datasets)

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\c24082331\.cache\huggingface\hub\datasets--UniversalCEFR--caes_es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████|

In [4]:
# Filter to keep only valid CEFR levels
filtered_data = spanish_data.filter(lambda x: x["cefr_level"] in CEFR_LEVELS)

Filter: 100%|██████████| 31355/31355 [00:00<00:00, 98668.44 examples/s] 


In [5]:
# Remove duplicate texts
df = filtered_data.to_pandas().drop_duplicates(subset="text", keep="first")
filtered_data = Dataset.from_pandas(df)

In [6]:
filtered_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 20837
})

In [7]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [8]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [9]:
# Tokenize the dataset
tokenized_data = filtered_data.map(preprocess, batched=True, remove_columns=filtered_data.column_names)

# Split Spanish into train/val
n = len(tokenized_data)
train_end = int(0.8 * n)
dev_end = int(0.9 * n)

ds_train = tokenized_data.select(range(0, train_end))
ds_dev   = tokenized_data.select(range(train_end, dev_end))
ds_test  = tokenized_data.select(range(dev_end, n))

Map: 100%|██████████| 20837/20837 [00:02<00:00, 7042.83 examples/s]


In [10]:
ds_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 16669
})

In [11]:
ds_dev

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2084
})

In [12]:
ds_test

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2084
})

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True)

Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [15]:
# Training args
args = TrainingArguments(
    output_dir="./eurobert_cefr_spanish_only",  
    num_train_epochs=3, 
    per_device_train_batch_size=2,              
    per_device_eval_batch_size=3,                
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,
    seed=42,
    learning_rate=3.6e-5,
    warmup_ratio=0.1,
    gradient_accumulation_steps=16,      
    optim="adamw_torch_fused",                   
    lr_scheduler_type="linear",                  
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    save_total_limit=1,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  
)

C:\Users\c24082331\AppData\Local\Temp\ipykernel_21832\2040595295.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
# Train on Spanish-only data
trainer.train()

# Evaluate on Dev set
trainer.evaluate()

# Save model, tokenizer, and trainer state
save_dir = "./eurobert_cefr_spanish_only/final_model"
trainer.save_model(save_dir)                    
tokenizer.save_pretrained(save_dir)            
trainer.state.save_to_json(os.path.join(save_dir, "trainer_state.json"))  

print(f"Model saved to {save_dir}")

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.268400,0.174402,0.968330,0.968402,0.969753,0.968330,0.990338,0.976190,0.983213,0.963668,0.977193,0.970383,0.982368,0.915493,0.947752,0.990610,0.990610,0.990610,0.890909,1.000000,0.942308,0.000000,0.000000,0.000000


KeyboardInterrupt: 

In [18]:
# Save model, tokenizer, and trainer state
save_dir = "./eurobert_cefr_spanish_only/final_model"
trainer.save_model(save_dir)                    
tokenizer.save_pretrained(save_dir)            
trainer.state.save_to_json(os.path.join(save_dir, "trainer_state.json"))  

print(f"Model saved to {save_dir}")

Model saved to ./eurobert_cefr_spanish_only/final_model
